## **Context**




We are going to use a large language model to automate the classification and processing of user help desk support tickets.  The ultimate goal would be to predict ticket categories, assign priority, suggest estimated resolution time, generate a response based on sentiment analysis from the LLM, and create output that is stored in a dataframe. The input file is Support_ticket_text_data.xls.

The dateframe should have 7 columns:

Support ticket ID (from input file), support ticket text (from input file), category, tags, priority, estimated resolution time, and a generated reply from the LLM.

You will likely need to run this from Google Colab.

## **Project Objective**

Develop a Generative AI application using a Large Language Model to **automate the classification and processing of support tickets.** The application will aim to predict ticket categories, assign priority, suggest estimated resolution times, generate responses based on sentiment analysis, and store the results in a structured DataFrame.


## **Model Loading**

In [11]:
# Installation for GPU llama-cpp-python
!pip install llama-cpp-python

     ---------------------------------------- 0.0/37.4 MB ? eta -:--:--
     ---------------------------------------- 0.0/37.4 MB ? eta -:--:--
     --------------------------------------- 0.0/37.4 MB 393.8 kB/s eta 0:01:35
     ---------------------------------------- 0.2/37.4 MB 1.3 MB/s eta 0:00:29
     ---------------------------------------- 0.4/37.4 MB 2.1 MB/s eta 0:00:18
      --------------------------------------- 0.7/37.4 MB 3.3 MB/s eta 0:00:12
     - -------------------------------------- 1.0/37.4 MB 3.9 MB/s eta 0:00:10
     - -------------------------------------- 1.5/37.4 MB 4.8 MB/s eta 0:00:08
     -- ------------------------------------- 2.4/37.4 MB 6.6 MB/s eta 0:00:06
     --- ------------------------------------ 2.9/37.4 MB 7.2 MB/s eta 0:00:05
     --- ------------------------------------ 3.1/37.4 MB 7.2 MB/s eta 0:00:05
     ---- ----------------------------------- 3.9/37.4 MB 7.7 MB/s eta 0:00:05
     ---- ----------------------------------- 4.4/37.4 MB 7.6 MB/


[notice] A new release of pip is available: 23.3.1 -> 24.0
[notice] To update, run: python.exe -m pip install --upgrade pip


In [12]:
# Install the hugging face hub
!pip install huggingface_hub -q


[notice] A new release of pip is available: 23.3.1 -> 24.0
[notice] To update, run: python.exe -m pip install --upgrade pip


### **Use Python code to import the 'hf_hub_download' function from the 'huggingface_hub' library and also imports the 'Llama' class from the 'llama_cpp' library.**


In [ ]:
#!sudo find /usr/ -name 'libcuda.so.*'

In [13]:
#needed
!pip install libcuda1

ERROR: Could not find a version that satisfies the requirement libcuda1 (from versions: none)
ERROR: No matching distribution found for libcuda1

[notice] A new release of pip is available: 23.3.1 -> 24.0
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
# We will use the 'hf_hub_download' function from the 'huggingface_hub' library

from huggingface_hub import hf_hub_download


C:\Users\codyd\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [16]:
#!sudo apt-get -y install cuda-12-0  # not used - needs reboot

#!nvcc --version
!pip install llama_cpp

ERROR: Could not find a version that satisfies the requirement llama_cpp (from versions: none)
ERROR: No matching distribution found for llama_cpp

[notice] A new release of pip is available: 23.3.1 -> 24.0
[notice] To update, run: python.exe -m pip install --upgrade pip


In [17]:
# We will use the 'Llama' class from the 'llama_cpp' library

from llama_cpp import Llama

In [18]:
# Define the model name or path as a string (You can find this info from hugging face website)

model_name_or_path = "TheBloke/Llama-2-13B-chat-GGUF"

# Define the model basename as a string, indicating it's in the gguf format

model_basename = "llama-2-13b-chat.Q5_K_M.gguf" # the model is in gguf format

In [19]:
# Download the model from the Hugging Face Hub using the 'hf_hub_download' function
# by specifying the 'repo_id' and 'filename'
model_path = hf_hub_download(
    repo_id=model_name_or_path,
    filename=model_basename
    )

KeyboardInterrupt: 

In [ ]:
# Create an instance of the 'Llama' class with specified parameters
# remove the blank spaces and complete the code

lcpp_llm = Llama(
        model_path=model_path,
        n_threads=2,  # CPU cores
        n_batch=512,  # Should be between 1 and n_ctx, consider the amount of VRAM in your GPU.
        n_gpu_layers=43,  # Change this value based on your model and your GPU VRAM pool.
        n_ctx=4096,  # Context window
    )

### **Define the 7 values noted in the initial project definition **

Write a Python function called **generate_llama_response** that takes the support_ticket_text column from the input dataset.  Your function needs to perform the following tasks:

Define a system message as a string and assign it to the variable system_message.

- **Combine the support_ticket_text and system_message to create a prompt string.**

*Generate a response from the LLaMA model using the lcpp_llm instance with the following parameters:*

- prompt should be the combined prompt string.
- max_tokens should be set to 256.
- temperature should be set to 0.
- top_p should be set to 0.95.
- repeat_penalty should be set to 1.2.
- top_k should be set to 50.
- stop should be set as a list containing 'INST'.
- echo should be set to False.
Extract and return the response text from the generated response.

**Provide a value for the system_message variable before using it in the function. **



*What content and instructions should be included in the system message to guide the technical assistant when processing support tickets? Please provide a detailed description of the information and guidelines that the system message should contain.*

Ideally, you need to provide information to a technical assistant that is processing a support ticket. The
breakdown of what should be included:

- **Introduction (System Role):** Begin with an introductory statement that establishes the role of the system message. In this case, it's acting as a guide for a technical assistant.

- **Ticket Categorization:** Explain the primary task of the technical assistant, which is to classify the support ticket into specific categories. In this example, the categories are:
    - Technical Issues
    - Hardware Issues
    - Data Recovery

- **Response Options:** Clearly state that the assistant should only respond with one of the predefined categories, emphasizing that other responses are not acceptable.

- **Sub-Tasks:** Outline the secondary tasks that the technical assistant should perform once the category is identified. These sub-tasks include:
  - **Creating Tags:** Instruct the assistant to create tags that will help further classify the ticket.
  - **Assigning Priority:** Specify that the assistant should assign a priority level (e.g., "High" or "Normal") based on their understanding of the text.
  - **Suggesting ETA:** Guide the assistant to provide an estimated time for
resolving the issue mentioned in the ticket.
  - **Generating 1st Reply (Sentiment-Based):** Emphasize the importance of crafting a response that aligns with the sentiment expressed in the ticket.

- **General Instructions:** Offer general instructions that should be followed throughout the ticket processing, such as:

  - **Categorization:** Reiterate that the assistant should categorize the ticket only into the predefined categories.
  - **Reading Carefully:** Stress the importance of reading the support ticket text thoroughly and considering the overall sentiment before responding.
  - **Tone:** Emphasize that the tone of all responses should be polite and professional.
  - **Output Format:** Clearly specify the desired output format for the responses generated by the assistant. In this case, the output should be in JSON format.

##### **The output of the model should be in JSON format**

The next two cells provide a general idea what the generate_llama_response function needs to look like. You will only have one function, but you will find ideas in both versions below.

In [ ]:
'''def generate_llama_response(support_ticket_text):

    # System message
    system_message = f"User: {user_prompt}\nSystem: "

    # Combine user_prompt and system_message to create the prompt
    prompt = f"{user_prompt}\n{system_message}"

    # Generate a response from the LLaMA model
    response = lcpp_llm(
        prompt=prompt,
        max_tokens=256,
        temperature=0,
        top_p=0.95,
        repeat_penalty=1.2,
        top_k=50,
        stop=['INST'],
        echo=False
    )

    # Extract and return the response text
    response_text = response["choices"][0]["text"]  ### Fill in the blank
    return response_text'''



In [ ]:
def generate_llama_response(support_ticket_text):

    # System message
    system_message = '''The role of the system message is to act as a guide for a technical assistant.

The primary task of the technical assistant is to classify the support ticket into specific categories. The categories are:

    1. Technical Issues
    2. Hardware Issues
    3. Data Recovery'''

    # user prompt
    user_prompt = '''General Instructions: Offer general instructions that should be followed throughout the ticket processing, such as:

    Categorization: Categorize the ticket only into the predefined categories.
    Reading Carefully: Stress the importance of reading the support ticket text thoroughly and considering the overall sentiment before responding.
    Tone: The tone of all responses should be polite and professional.
    Output Format: The output should be in JSON format.'''



    # Combine user_prompt and system_message to create the prompt
    prompt = f"{user_prompt}\n{system_message}"

    # Generate a response from the LLaMA model
    response = lcpp_llm(
        prompt=prompt,
        max_tokens=256,
        temperature=0,
        top_p=0.95,
        repeat_penalty=1.2,
        top_k=50,
        stop=['INST'],
        echo=False
    )

    # Extract and return the response text
    response_text = response["choices"][0]["text"]  ### Fill in the blank
    return response_text
    print(prompt)

### **Load the input dataset into a dataframe)**

In [ ]:
# Import the pandas library and alias it as 'pd'
import pandas as pd


In [ ]:

# Read a CSV file into a DataFrame and store it in the 'data' variable
# import the reviews into a dataframe

# Mount Google drive to access the dataset
from google.colab import drive
drive.mount('/content/drive')

# Install the openpyxl library
!pip install openpyxl

# Read the XLS file into a DataFrame
data = pd.read_excel('/content/drive/MyDrive/ColabFiles/ARTI-330/Support_ticket_text_data.xls')

In [ ]:
# Number of records and columns
data.shape

In [ ]:
# Display part of the dataframe
data.head()

### **Create a new column in the DataFrame called 'llama_response' and populate it with responses generated by applying the 'generate_llama_response' function to each 'support_ticket_text' in the DataFrame**

In [ ]:
# Create new column in DataFrame
data['llama_response'] = data['support_ticket_text'].apply(lambda x: generate_llama_response(x))
print(data)

In [ ]:
# Apply a function to each element in the 'support_ticket_text' column of the DataFrame 'data'
# The applied function, in this case, is a lambda function.

# The lambda function takes a single argument 'x', representing each individual 'support_ticket_text' in the column.

# Inside the lambda function:
# - 'generate_llama_response(x)' is called to generate a response based on the 'support_ticket_text'.
# - The result of 'generate_llama_response(x)' is assigned to a new column called 'llama_response' in the DataFrame 'data'.

# Example - data['llama_response'] = data['support_ticket_text'].apply(lambda x: generate_llama_response(x))

In [ ]:
## Check the new_column added
data.head()

In [ ]:
data['llama_response'][0]

### **Create a report**

- You are free to utilize a different technique, some potential options below

In [ ]:
import json

In [ ]:
# Function to parse JSON data and extract key-value pairs
def extract_json_data(json_str):
    try:
        data_dict = json.loads(json_str)
        return data_dict
    except json.JSONDecodeError as e:
        print(f"Error parsing JSON: {e}")
        return {}

# Apply the function to the 'llama_response' column
data['llama_response_parsed'] = data['llama_response'].apply(extract_json_data)

In [ ]:
data

In [ ]:
data['llama_response_parsed'][0]

In [ ]:
# Concatenate the original DataFrame 'data' with a new DataFrame created by normalizing JSON data.

# The 'data['llama_response_parsed']' column is assumed to contain JSON data that needs to be flattened and normalized.

# 'pd.json_normalize' is a pandas function used to normalize semi-structured JSON data into a flat DataFrame.
# In this case, it's applied to the 'llama_response_parsed' column, which presumably contains JSON data.

# The result of 'pd.json_normalize' is a DataFrame where the JSON data is flattened and each element becomes a separate column.

# The 'axis=1' parameter specifies that the concatenation should be done horizontally, i.e., the new columns from normalization
# will be added as new columns in the 'data' DataFrame.

# After this operation, the 'data' DataFrame will contain the original columns along with the additional columns
# generated by normalizing the JSON data from the 'llama_response_parsed


# example - data = pd.concat([data, pd.json_normalize(data['llama_response_parsed'])], axis=1)


In [ ]:
data

In [ ]:
# Drop specific columns which are not needed from the DataFrame 'data'
# Keep the columns which are mentioned in the sample output

In [ ]:
data.head()

Include a brief summary

In [ ]:
data['llama_response'][1]



---

